# Multi-Tool AI Agent for Bangladesh

This notebook:
1. Installs dependencies
2. Clones the project repo (or uses inline code)
3. Downloads HuggingFace datasets → SQLite DBs
4. Runs the AI agent interactively

---

## 1️⃣ Install Dependencies

In [ ]:
%%capture
!pip install langchain langchain-core langchain-community langchain-google-genai \
             datasets pandas ddgs python-dotenv pydantic tavily-python langgraph


## 2️⃣ Clone Repository

In [ ]:
!git clone https://github.com/MuktoFlame/bd-data-assistant.git
%cd bd-data-assistant

## 3️⃣ Set API Keys

In [ ]:
import os

# ── Set Google AI Studio key ──────────────────────────────────────────────────
os.environ['GOOGLE_API_KEY'] = 'your-google-api-key-here'         # Get from aistudio.google.com

# ── Web Search (optional — DuckDuckGo is the free fallback) ──────────────────
# os.environ['TAVILY_API_KEY']  = 'your-tavily-api-key-here'
# os.environ['SERPAPI_API_KEY'] = 'your-serpapi-api-key-here'

print('✅ API keys set')


## 4️⃣ Download Datasets & Build SQLite Databases

In [ ]:
!python setup_databases.py

## 5️⃣ Verify Databases

In [ ]:
import sqlite3, pandas as pd

for db, table in [('data/institutions.db','institutions'),
                   ('data/hospitals.db','hospitals'),
                   ('data/restaurants.db','restaurants')]:
    con = sqlite3.connect(db)
    df  = pd.read_sql(f'SELECT * FROM {table} LIMIT 3', con)
    con.close()
    print(f'\n📦 {db} — {table}:')
    display(df)

## 6️⃣ Create & Run the Agent

In [ ]:
from agent.main_agent import build_agent
agent = build_agent()
print('✅ Agent ready!')


## 7️⃣ Example Queries

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# ── DB queries ────────────────────────────────────────────────────────────────
queries = [
    # Hospitals
    'How many hospitals are in Dhaka?',
    'Which hospital in Sylhet has the most beds?',
    'List all government hospitals in Rajshahi.',

    # Institutions
    'How many public universities are there in Bangladesh?',
    'List educational institutions in Chittagong.',

    # Restaurants
    'Top-rated restaurants in Cox\'s Bazar?',
    'How many restaurants serve Chinese cuisine?',

    # Web search
    'What is the role of DGHS in Bangladesh?',
    'What is the healthcare policy in Bangladesh?',
]

for q in queries[:3]:   # run first 3 to save time; remove slice for all
    print(f'\n🔹 Query: {q}')
    print('-' * 60)
    result = agent.invoke({'messages': [HumanMessage(content=q)]})
    output_messages = result.get('messages', [])
    answer = ''
    for msg in reversed(output_messages):
        if isinstance(msg, AIMessage) and msg.content:
            content = msg.content
            if isinstance(content, list):
                answer = '\n'.join([b['text'] for b in content if isinstance(b, dict) and b.get('type') == 'text'])
            else:
                answer = str(content)
            break
    print(answer)
    print()


## 8️⃣ Interactive Chat Loop

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
print('Type your question (or "exit" to stop):\n')

while True:
    user_input = input('You: ').strip()
    if not user_input or user_input.lower() in {'exit', 'quit'}:
        print('Session ended.')
        break
    messages = list(chat_history) + [HumanMessage(content=user_input)]
    result = agent.invoke({'messages': messages})
    output_messages = result.get('messages', [])
    answer = ''
    for msg in reversed(output_messages):
        if isinstance(msg, AIMessage) and msg.content:
            content = msg.content
            if isinstance(content, list):
                answer = '\n'.join([b['text'] for b in content if isinstance(b, dict) and b.get('type') == 'text'])
            else:
                answer = str(content)
            break
    print(f'Agent: {answer}\n')
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=answer))
